In [1]:
import os
import math
import random
import types
from pathlib import Path

import torch
import networkx as nx

torch.manual_seed(7)
random.seed(7)

#BASE = Path("/mnt/data")
#print("Files in /mnt/data:", sorted(p.name for p in BASE.iterdir() if p.is_file())[:20])


c:\Users\zly18\anaconda3\Lib\site-packages\networkx\utils\backends.py:102: RuntimeWarning: networkx backend name is not a valid identifier: 'nx-loopback'. Ignoring.
  backends = _get_backends("networkx.backends")


In [22]:
import networkx as nx
import cordon_environment
import deepaco
import importlib
importlib.reload(cordon_environment)
importlib.reload(deepaco)
from cordon_environment import CordonEnv
from deepaco import DeepACOAgent

In [25]:
road_graph = nx.Graph()
road_graph.add_nodes_from([1, 2, 3, 4, 5, 6])
road_graph.add_edges_from([
    (1, 2),
    (1, 3),
    (2, 4),
    (3, 4),
    (4, 5),
    (5, 6),
    (3, 6),
])

pos = {
    1: (0, 0),
    2: (1, 1),
    3: (1, -1),
    4: (2, 0),
    5: (3, 0),
    6: (2, -2),
}
print("nodes:", list(road_graph.nodes()))
print("edges:", list(road_graph.edges()))


nodes: [1, 2, 3, 4, 5, 6]
edges: [(1, 2), (1, 3), (2, 4), (3, 4), (3, 6), (4, 5), (5, 6)]


In [26]:
all_nodes = [-1] + sorted(list(road_graph.nodes()))
node2idx = {n: i for i, n in enumerate(all_nodes)}

n = len(all_nodes)
heu_mat = torch.full((n, n), 0.05, dtype=torch.float32)

# Real graph edges get stronger heuristic
for u, v in road_graph.edges():
    i, j = node2idx[u], node2idx[v]
    heu_mat[i, j] = 1.5
    heu_mat[j, i] = 1.5

# Virtual node connects to all real nodes
for u in road_graph.nodes():
    i, j = node2idx[-1], node2idx[u]
    heu_mat[i, j] = 2.0
    heu_mat[j, i] = 1.0

print("all_nodes:", all_nodes)
print("node2idx:", node2idx)
print("heuristic matrix shape:", tuple(heu_mat.shape))


all_nodes: [-1, 1, 2, 3, 4, 5, 6]
node2idx: {-1: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6}
heuristic matrix shape: (7, 7)


In [27]:
class ConstantReward:
    def __call__(self, state, policy_kind=None, policy_value=None):
        # Can return float directly, which is accepted by the agent.
        return 1.0
    def evaluate(self, zone_state, policy_kind=None, policy_value=None):
        return 1.0

reward_fn = ConstantReward()
print("dummy reward on [-1, 1, 2] =", reward_fn([-1, 1, 2]))


dummy reward on [-1, 1, 2] = 1.0


In [28]:
agent = DeepACOAgent(
    node2idx=node2idx,
    n_ants=5,
    alpha=1.0,
    beta=1.0,
    decay=0.9,
    max_steps=6,
    virtual_node=-1,
    elitist=False,
    min_max=False,
    pheromone_init=1.0,
    heuristic_floor=1e-12,
    device="cpu",
)

print("pheromone shape:", tuple(agent.pheromone.shape))
print("initial pheromone mean:", float(agent.pheromone.mean()))


pheromone shape: (7, 7)
initial pheromone mean: 1.0


In [29]:
env = CordonEnv(road_graph=road_graph, max_steps=6, virtual_node=-1)
env.reset()

print("step 0")
print("env.path =", env.path)
print("env.available_actions() =", env.available_actions())
print("agent._state_available_actions(env) =", agent._state_available_actions(env))

# Move to one real node
env.step(1)
print("\nafter choosing 1")
print("env.path =", env.path)
print("env.available_actions() =", env.available_actions())
print("agent._state_available_actions(env) =", agent._state_available_actions(env))

# Move to another real node
env.step(2)
print("\nafter choosing 4")
print("env.path =", env.path)
print("env.available_actions() =", env.available_actions())
print("agent._state_available_actions(env) =", agent._state_available_actions(env))


step 0
env.path = [-1]
env.available_actions() = [1, 2, 3, 4, 5, 6]
agent._state_available_actions(env) = [1, 2, 3, 4, 5, 6]

after choosing 1
env.path = [-1, 1]
env.available_actions() = [2, 3, -1]
agent._state_available_actions(env) = [2, 3, -1]

after choosing 4
env.path = [-1, 1, 2]
env.available_actions() = [3, 4, -1]
agent._state_available_actions(env) = [3, 4, -1]


In [30]:
env = CordonEnv(road_graph=road_graph, max_steps=6, virtual_node=-1)
env.reset()
action, support_edge, log_prob, entropy = agent.select_action(env, heu_mat, inference=False)
print("selected action:", action)
print("log_prob:", float(log_prob))
print("entropy:", float(entropy))


selected action: 4
log_prob: -1.7917594909667969
entropy: 1.791759729385376


In [31]:
env = CordonEnv(road_graph=road_graph, max_steps=6, virtual_node=-1)
rollout = agent.rollout(
    env=env,
    heu_mat=heu_mat,
    reward_fn=reward_fn,
    max_steps=6,
    terminal_reward_only=True,
    inference=False,
)

print("rollout.path =", rollout.path)
print("rollout.actions =", rollout.actions)
print("rollout.rewards =", rollout.rewards)
print("rollout.total_reward =", rollout.total_reward)
print("num log_probs =", len(rollout.log_probs))
print("num entropies =", len(rollout.entropies))


rollout.path = [-1, 1, 3, 4, 2, -1]
rollout.actions = [1, 3, 4, 2, -1]
rollout.rewards = 1.0
rollout.total_reward = 1.0
num log_probs = 5
num entropies = 5


## Multi-ant sampling

In [ ]:
def env_factory():
    return CordonEnv(road_graph=road_graph, max_steps=6, virtual_node=-1) # 

costs, log_probs, paths, entropies, rollouts = agent.sample(
    env_factory=env_factory,
    heu_mat=heu_mat,
    reward_fn=reward_fn,
    inference=False,
    terminal_reward_only=True,
)

print("sampled costs:", costs)
print("sampled paths:")
for i, p in enumerate(paths):
    print(f"  ant {i}: {p}")
print("number of ants:", len(paths))


sampled costs: [1.0, 1.0, 1.0, 1.0, 1.0]
sampled paths:
  ant 0: [-1, 3, 6, 5, 4, -1]
  ant 1: [-1, 2, 1, 4, 5, 3, -1]
  ant 2: [-1, 4, 2, 3, 5, 6, -1]
  ant 3: [-1, 2, 1, 3, 4, 5, 6]
  ant 4: [-1, 6, 3, 1, 4, 2, -1]
number of ants: 5


## Pheromone update after sampling

In [33]:
before_mean = float(agent.pheromone.mean())
agent.update_pheromone(rollouts)
after_mean = float(agent.pheromone.mean())

print("pheromone mean before:", before_mean)
print("pheromone mean after :", after_mean)
print("pheromone matrix:\n", agent.pheromone)


pheromone mean before: 1.0
pheromone mean after : 1.4918369054794312
pheromone matrix:
 tensor([[0.9000, 0.9000, 2.9000, 1.9000, 1.9000, 0.9000, 1.9000],
        [0.9000, 0.9000, 1.9000, 1.9000, 0.9000, 0.9000, 0.9000],
        [1.9000, 2.9000, 0.9000, 0.9000, 1.9000, 0.9000, 0.9000],
        [1.9000, 1.9000, 0.9000, 0.9000, 3.9000, 0.9000, 2.9000],
        [2.9000, 0.9000, 1.9000, 2.9000, 0.9000, 3.9000, 0.9000],
        [0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 0.9000, 1.9000],
        [0.9000, 0.9000, 0.9000, 1.9000, 0.9000, 1.9000, 0.9000]])


## Deterministic inference mode

In inference mode, the agent selects the argmax action instead of sampling.


In [34]:
env = CordonEnv(road_graph=road_graph, max_steps=6, virtual_node=-1)
rollout_inf = agent.rollout(
    env=env,
    heu_mat=heu_mat,
    reward_fn=reward_fn,
    max_steps=6,
    terminal_reward_only=True,
    inference=True,
)

print("inference rollout path:", rollout_inf.path)
print("inference rollout actions:", rollout_inf.actions)
print("inference total reward:", rollout_inf.total_reward)


inference rollout path: [-1, 2, 4, 3, 1, 6, 5]
inference rollout actions: [2, 4, 3, 1, 6, 5]
inference total reward: 1.0


## Conclusion

This notebook verifies that the simplified ACO agent can:

- generate actions from the **whole state/path frontier**
- select actions
- execute a single rollout
- execute multi-ant sampling
- update pheromone

The reward function here is only a placeholder.  
You can later replace `ConstantReward` with your real `MSARewardFunction`.
